In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [4]:
RAW_PATH = '../data/accepted_2007_to_2018Q4.csv.gz'

In [5]:
df_raw = pd.read_csv(
    RAW_PATH,
    
    low_memory=False,
    nrows = 300000,
)

print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

Loaded: 300,000 rows × 151 columns


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,...,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.00,3600.00,3600.00,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.00,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,...,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.00,24700.00,24700.00,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.00,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,...,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.00,20000.00,20000.00,60 months,10.78,432.66,B,B4,truck driver,10+ years,MORTGAGE,63000.00,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,home_improvement,NaN,605xx,IL,10.78,...,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
KEEP_COLS = [
    'id', 'loan_amnt', 'funded_amnt', 'term',
    'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc',
    'verification_status', 'issue_d', 'loan_status',
    'purpose', 'dti', 'delinq_2yrs',
    'fico_range_low', 'fico_range_high',
    'open_acc', 'revol_util', 'total_acc'
]

df = df_raw[KEEP_COLS].copy()
print(f"Working dataframe: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Working dataframe: 300,000 rows × 22 columns
Memory usage: 184.6 MB


In [7]:
print("All loan_status values:")
print(df['loan_status'].value_counts())

RESOLVED = ['Fully Paid', 'Charged Off']
df = df[df['loan_status'].isin(RESOLVED)].copy()

print(f"\nAfter filtering: {df.shape[0]:,} rows")
print(df['loan_status'].value_counts(normalize=True).mul(100).round(1))


All loan_status values:
loan_status
Fully Paid            212275
Charged Off            53501
Current                32462
Late (31-120 days)      1063
In Grace Period          483
Late (16-30 days)        215
Default                    1
Name: count, dtype: int64

After filtering: 265,776 rows
loan_status
Fully Paid    79.90
Charged Off   20.10
Name: proportion, dtype: float64


In [8]:
# TARGET: 1 = defaulted (Charged Off), 0 = repaid (Fully Paid)
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

print(df['default'].value_counts())
print(f"Default rate: {df['default'].mean()*100:.1f}%")

default
0    212275
1     53501
Name: count, dtype: int64
Default rate: 20.1%


In [9]:
def to_numeric_safe(series):
    """Strip % if present, then cast to float. Works on both str and numeric columns."""
    if series.dtype == object:  # it's a string column
        series = series.str.replace('%', '', regex=False).str.strip()
    return pd.to_numeric(series, errors='coerce')

# int_rate and revol_util — strip % only if needed
df['int_rate']   = to_numeric_safe(df['int_rate'])
df['revol_util'] = to_numeric_safe(df['revol_util'])

# term — extract integer (36 or 60) from " 36 months" or already int
if df['term'].dtype == object:
    df['term'] = df['term'].str.strip().str.extract(r'(\d+)').astype(int)

# issue_d — parse to datetime
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# fico midpoint
df['fico_mid'] = (df['fico_range_low'] + df['fico_range_high']) / 2

# confirm final dtypes
print(df[['int_rate', 'revol_util', 'term', 'issue_d', 'fico_mid']].dtypes)

int_rate             float64
revol_util           float64
term                   int64
issue_d       datetime64[ns]
fico_mid             float64
dtype: object


In [10]:
# cap revol_util at 100
df['revol_util'] = df['revol_util'].clip(upper=100)

# cap annual_inc at 99th percentile
inc_cap = df['annual_inc'].quantile(0.99)
df['annual_inc'] = df['annual_inc'].clip(upper=inc_cap)

# fill emp_length nulls
df['emp_length'] = df['emp_length'].fillna('Unknown')

# null summary
null_pct = df.isnull().mean().mul(100).round(2)
print(null_pct[null_pct > 0].sort_values(ascending=False))

revol_util   0.04
dtype: float64


In [11]:
print("=== Dataset summary ===")
print(f"Rows:         {df.shape[0]:,}")
print(f"Columns:      {df.shape[1]}")
print(f"Default rate: {df['default'].mean()*100:.1f}%")
print(f"Date range:   {df['issue_d'].min().date()} → {df['issue_d'].max().date()}")
print(f"Avg loan:     ${df['loan_amnt'].mean():,.0f}")
print(f"Grades:       {sorted(df['grade'].unique())}")

# Save cleaned file so you don't redo this each session
df.to_csv('../outputs/cleaned_loans.csv', index=False)
print("\nSaved to outputs/cleaned_loans.csv")

=== Dataset summary ===
Rows:         265,776
Columns:      24
Default rate: 20.1%
Date range:   2015-04-01 → 2015-12-01
Avg loan:     $14,579
Grades:       ['A', 'B', 'C', 'D', 'E', 'F', 'G']

Saved to outputs/cleaned_loans.csv
